please refer to [Raw](../data/raw/raw.md)

In [1]:
from pathlib import Path
import re

import pandas as pd

pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.3f}".format

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "milan_crashes_monthly": {
        "file": RAW_DIR / "milan_crashes_monthly.csv",
        "id_cols": ["Anno", "Mese"],
    },
    "milan_crashes_monthly_city_ring": {
        "file": RAW_DIR / "milan_crashes_monthly_city_ring.csv",
        "id_cols": ["Anno", "Mese", "Cerchia"],
    },
    "milan_crashes_by_nature": {
        "file": RAW_DIR / "milan_crashes_by_nature.csv",
        "id_cols": ["Anno", "Mese", "NaturaIncidente"],
    },
    "milan_crashes_by_vehicles": {
        "file": RAW_DIR / "milan_crashes_by_vehicles.csv",
        "id_cols": ["Anno", "Mese"],
    },
    "milan_crashes_by_zone": {
        "file": RAW_DIR / "milan_crashes_by_zone.csv",
        "id_cols": ["Anno", "Mese", "Municipio"],
    },
}


def load_raw_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8-sig")

In [2]:
# Cleaning function each dataset can be run through
def clean_common(df: pd.DataFrame, id_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()

    id_cols_set = set(id_cols)
    special_cols = {"Anno", "Mese"}

    for col in out.columns:
        # Convert "Anno" and "Mese" to numeric
        if col in special_cols:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
            continue

        # Check if the column is string/object
        is_str_or_obj = pd.api.types.is_string_dtype(
            out[col]
        ) or pd.api.types.is_object_dtype(out[col])

        # If ID is string, set NA
        if col in id_cols_set:
            if is_str_or_obj:
                out[col] = out[col].astype("string").str.strip().replace("", pd.NA)
            continue

        # All other string columns, set NA + attempt numeric conversion
        if is_str_or_obj:
            s_clean = out[col].astype("string").str.strip().replace("", pd.NA)

            converted = pd.to_numeric(s_clean, errors="coerce")

            if converted.notna().mean() > 0.8:
                out[col] = converted
            else:
                out[col] = s_clean
    
    # Create "month_start" datetime column if "Anno" and "Mese" are present
    if special_cols.issubset(out.columns):
        out["month_start"] = pd.to_datetime(
            {
                "year": out["Anno"],
                "month": out["Mese"],
                "day": 1,
            },
            errors="coerce",
        )

    return out

In [3]:
# Replace spaces with a single space, remove spaces around dashes
def normalize_natura_incidente(series: pd.Series) -> pd.Series:
    out = series.astype("string").str.strip()
    out = out.str.replace(r"\s+", " ", regex=True)
    out = out.str.replace(r"\s*-\s*", "-", regex=True)
    return out

In [4]:
def clean_dataset(name: str, cfg: dict) -> tuple[pd.DataFrame, dict]:
    raw = load_raw_csv(cfg["file"])
    before_rows = len(raw)

    cleaned = clean_common(raw, cfg["id_cols"])

    # Keep unknown IDs as NA for consistency across datasets.
    if name == "milan_crashes_by_zone":
        cleaned["Municipio"] = pd.to_numeric(
            cleaned["Municipio"], errors="coerce"
        ).astype("Int64")

    if name == "milan_crashes_by_nature":
        cleaned["NaturaIncidente"] = normalize_natura_incidente(
            cleaned["NaturaIncidente"]
        )

    numeric_cols = [
        c
        for c in cleaned.columns
        if c not in cfg["id_cols"]
        and c not in {"month_start"}
        and pd.api.types.is_numeric_dtype(cleaned[c])
    ]
    for col in numeric_cols:
        cleaned[col] = cleaned[col].round().astype("Int64")

    cleaned = (
        cleaned.drop_duplicates()
        .sort_values(
            [c for c in ["Anno", "Mese"] if c in cleaned.columns]
            + [c for c in cfg["id_cols"] if c not in {"Anno", "Mese"}],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    qc = {
        "rows_before": before_rows,
        "rows_after": len(cleaned),
        "duplicates_removed": before_rows - len(cleaned),
        "missing_cells": int(cleaned.isna().sum().sum()),
    }
    return cleaned, qc

In [5]:
cleaned_data = {}
qc_rows = []

for name, cfg in DATASETS.items():
    cleaned_df, qc = clean_dataset(name, cfg)
    cleaned_data[name] = cleaned_df

    invalid_month_rows = (
        int((~cleaned_df["Mese"].between(1, 12)).fillna(True).sum())
        if "Mese" in cleaned_df.columns
        else 0
    )

    num_df = cleaned_df.select_dtypes(include=["number"])
    negative_numeric_cells = int((num_df < 0).sum().sum()) if not num_df.empty else 0

    qc_rows.append(
        {
            "dataset": name,
            "rows_before": qc["rows_before"],
            "rows_after": qc["rows_after"],
            "duplicates_removed": qc["duplicates_removed"],
            "invalid_month_rows": invalid_month_rows,
            "negative_numeric_cells": negative_numeric_cells,
            "missing_cells": int(cleaned_df.isna().sum().sum()),
        }
    )

qc_scan = pd.DataFrame(qc_rows).sort_values("dataset").reset_index(drop=True)
qc_scan

,dataset,rows_before,rows_after,duplicates_removed,invalid_month_rows,negative_numeric_cells,missing_cells
0,milan_crashes_by_nature,2590,2590,0,0,0,0
1,milan_crashes_by_vehicles,288,288,0,0,0,174
2,milan_crashes_by_zone,2880,2880,0,0,0,288
3,milan_crashes_monthly,288,288,0,0,0,3
4,milan_crashes_monthly_city_ring,1440,1440,0,0,0,0


In [6]:
monthly = cleaned_data["milan_crashes_monthly"].copy()
monthly["IncidentiTotali"] = monthly["IncidentiMortali"] + monthly["IncidentiSoliFeriti"]

In [7]:
def compare_to_monthly(
    reference: pd.DataFrame,
    other: pd.DataFrame,
    value_col: str,
    label: str,
) -> pd.DataFrame:
    ref = reference[["Anno", "Mese", "IncidentiTotali"]].rename(
        columns={"IncidentiTotali": "incidenti_monthly"}
    )
    agg = other.groupby(["Anno", "Mese"], as_index=False).agg(
        incidenti_other=(value_col, "sum")
    )
    out = ref.merge(agg, on=["Anno", "Mese"], how="left")
    out["incidenti_other"] = out["incidenti_other"].fillna(0)
    out["delta"] = out["incidenti_other"] - out["incidenti_monthly"]
    out["source"] = label
    return out

In [8]:
vehicles_cols = [
    c
    for c in cleaned_data["milan_crashes_by_vehicles"].columns
    if re.match(r"^incidenti", c)
]
cleaned_data["milan_crashes_by_vehicles"]["incidenti_tot_veicoli"] = (
    cleaned_data["milan_crashes_by_vehicles"][vehicles_cols].sum(axis=1, min_count=1)
)

recon_tables = [
    compare_to_monthly(monthly, cleaned_data["milan_crashes_by_nature"], "Incidenti", "by_nature"),
    compare_to_monthly(monthly, cleaned_data["milan_crashes_monthly_city_ring"], "Incidenti", "city_ring"),
    compare_to_monthly(monthly, cleaned_data["milan_crashes_by_zone"], "Incidenti", "by_zone"),
    compare_to_monthly(monthly, cleaned_data["milan_crashes_by_vehicles"], "incidenti_tot_veicoli", "by_vehicles"),
]

reconciliation = pd.concat(recon_tables, ignore_index=True)

reconciliation_summary = (
    reconciliation.groupby("source", as_index=False)
    .agg(
        months_compared=("delta", "size"),
        months_with_reference=("incidenti_monthly", lambda s: int(s.notna().sum())),
        months_perfect_match=("delta", lambda s: int((s.dropna() == 0).sum())),
        max_abs_delta=("delta", lambda s: int(s.abs().max(skipna=True) if s.notna().any() else 0)),
    )
    .sort_values("source")
    .reset_index(drop=True)
)
reconciliation_summary["months_missing_reference"] = (
    reconciliation_summary["months_compared"] - reconciliation_summary["months_with_reference"]
)

reconciliation_summary

,source,months_compared,months_with_reference,months_perfect_match,max_abs_delta,months_missing_reference
0,by_nature,288,285,285,0,3
1,by_vehicles,288,285,285,0,3
2,by_zone,288,285,285,0,3
3,city_ring,288,285,285,0,3


In [9]:
for name, df in cleaned_data.items():
    out_file = PROCESSED_DIR / f"{name}_cleaned.csv"
    df.to_csv(out_file, index=False)

monthly_features = monthly.copy()
monthly_features["morti_per_100_incidenti"] = 100 * monthly_features["Morti"] / monthly_features["IncidentiTotali"]
monthly_features["feriti_per_incidente"] = monthly_features["Feriti"] / monthly_features["IncidentiTotali"]
monthly_features["is_summer"] = monthly_features["Mese"].isin([6, 7, 8]).astype(int)
monthly_features["is_winter"] = monthly_features["Mese"].isin([12, 1, 2]).astype(int)

monthly_features.to_csv(PROCESSED_DIR / "milan_crashes_monthly_features.csv", index=False)

print(f"Saved cleaned datasets in: {PROCESSED_DIR.resolve()}")

Saved cleaned datasets in: /Users/faustozamparelli/Documents/Developer/MilanCrash/data/processed
